# Double Descent

The classical narrative of generalization — that model complexity has an optimal sweet spot, found at the minimum of a U-shaped test error curve — has guided statistical learning practice for decades. It is mathematically precise, pedagogically clean, and [wrong for modern neural networks]{.mark}.

Belkin et al. [@belkin2019] and Nakkiran et al. [@nakkiran2020] showed that as model capacity continues to grow past the point where training error reaches zero, test error does not stay high — it descends a second time. The resulting curve has two descent phases separated by a peak at the **interpolation threshold**: the critical capacity at which the model can just barely memorize the training data. This is the **double descent** phenomenon.

In this appendix we develop the theory from the classical bias-variance tradeoff through both the model-wise and epoch-wise forms of double descent, and then reproduce the core experimental curves from Nakkiran et al. on CIFAR-10 with ResNets of varying width.

<br>

In [ ]:
#| echo: false
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
import warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib_inline import backend_inline

import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import CosineAnnealingLR

DATASET_DIR   = Path("./data").resolve()
ARTIFACTS_DIR = Path("./artifacts").resolve()
DATASET_DIR.mkdir(exist_ok=True)
ARTIFACTS_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42
DEBUG = False
MATPLOTLIB_FORMAT = "png" if DEBUG else "svg"

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
warnings.simplefilter(action="ignore")
backend_inline.set_matplotlib_formats(MATPLOTLIB_FORMAT)

DEVICE = (
    torch.device("cuda:0") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Device: {DEVICE}")

## Classical Bias-Variance Recap

Consider a regression problem where the true relationship between input $x$ and output $y$ is $y = f(x) + \varepsilon$ with $\varepsilon \sim \mathcal{N}(0, \sigma^2).$ A learning algorithm trained on a finite dataset $\mathcal{D}$ produces an estimator $\hat{f}_\mathcal{D}(x).$ The expected squared prediction error at a test point $x$ decomposes as:

$$
\mathbb{E}_{\mathcal{D}, \varepsilon}\left[(y - \hat{f}_\mathcal{D}(x))^2\right]
= \underbrace{\left(\mathbb{E}_\mathcal{D}[\hat{f}_\mathcal{D}(x)] - f(x)\right)^2}_{\text{Bias}^2[\hat{f}(x)]}
+ \underbrace{\mathbb{E}_\mathcal{D}\left[\left(\hat{f}_\mathcal{D}(x) - \mathbb{E}_\mathcal{D}[\hat{f}_\mathcal{D}(x)]\right)^2\right]}_{\text{Var}[\hat{f}(x)]}
+ \sigma^2.
$$

The **bias** measures how far the average prediction is from the truth; it decreases as the model family becomes more expressive. The **variance** measures how sensitive the predictor is to the particular training set drawn; it increases with model complexity since a more flexible model fits the noise in each sample differently. The irreducible noise $\sigma^2$ is a property of the data-generating process and cannot be reduced by any model.

The classical picture, borne out clearly in polynomial regression and $k$-nearest neighbors, is that test error forms a U-shaped curve as a function of model complexity: a sweet spot exists where bias and variance are balanced. For a full statistical treatment of this decomposition see NB01 in the Classical ML series.

We reproduce this curve using polynomial regression on a noisy sinusoid. For each degree $d$ we fit 50 random train/test splits and record the mean squared error on each.

Generating the bias-variance tradeoff curve across polynomial degrees:

In [ ]:
#| label: fig-bias-variance-classical
#| fig-cap: "Classical bias-variance tradeoff. Train error (blue) decreases monotonically with polynomial degree. Test error (orange) follows a U-shape: underfitting at low degree, overfitting at high degree. Shaded bands show ±1 standard deviation across 50 random splits. The dashed vertical line marks the degree with lowest mean test error."
#| code-fold: true

rng = np.random.default_rng(0)

N_TOTAL  = 200
N_TRAIN  = 40
N_SPLITS = 50
MAX_DEG  = 20
SIGMA    = 0.4

# Ground truth: noisy sinusoid
x_all = rng.uniform(-1, 1, size=N_TOTAL)
y_all = np.sin(2 * np.pi * x_all) + rng.normal(scale=SIGMA, size=N_TOTAL)

degrees   = np.arange(1, MAX_DEG + 1)
train_mse = np.zeros((N_SPLITS, MAX_DEG))
test_mse  = np.zeros((N_SPLITS, MAX_DEG))

for s in range(N_SPLITS):
    idx = rng.permutation(N_TOTAL)
    tr, te = idx[:N_TRAIN], idx[N_TRAIN:]
    x_tr, y_tr = x_all[tr], y_all[tr]
    x_te, y_te = x_all[te], y_all[te]

    for i, d in enumerate(degrees):
        coeffs = np.polyfit(x_tr, y_tr, d)           # <1>
        p = np.poly1d(coeffs)
        train_mse[s, i] = np.mean((p(x_tr) - y_tr) ** 2)
        test_mse[s, i]  = np.mean((p(x_te) - y_te) ** 2)

mean_tr = train_mse.mean(0)
std_tr  = train_mse.std(0)
mean_te = test_mse.mean(0)
std_te  = test_mse.std(0)

best_deg = degrees[mean_te.argmin()]                  # <2>

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(degrees, mean_tr, color="C0", linewidth=2, label="train error")
ax.fill_between(degrees, mean_tr - std_tr, mean_tr + std_tr, color="C0", alpha=0.2)

ax.plot(degrees, mean_te, color="C1", linewidth=2, label="test error")
ax.fill_between(degrees, mean_te - std_te, mean_te + std_te, color="C1", alpha=0.2)

ax.axvline(best_deg, color="gray", linestyle="dashed", lw=1.2, label=f"optimal degree ({best_deg})")
ax.set_xlabel("polynomial degree")
ax.set_ylabel("MSE")
ax.set_ylim(0, 0.8)
ax.legend()
ax.grid(linestyle="dotted", alpha=0.6)
fig.tight_layout()
plt.show();

Implementation notes:

1. `np.polyfit` solves for the least-squares polynomial coefficients, which is the correct maximum-likelihood estimator for Gaussian noise.
2. The optimal degree minimizes mean test MSE across splits — our empirical proxy for expected generalization error.

**Figure.** The U-shaped test error curve is clearly visible. Train error decreases monotonically as degree increases; the model has enough parameters to fit the training data exactly for high degrees (interpolating through all $N_{\text{train}} = 40$ points with a degree-40 polynomial). Test error, however, rises sharply at high degrees because the high-degree polynomials fit the noise rather than the signal.

This picture is self-consistent and correct — within the regime of models that *cannot* perfectly interpolate the training data. The classical story breaks down when models are powerful enough to memorize the training set, which is precisely the regime of modern deep learning. As Belkin et al. [@belkin2019] observed, something qualitatively different happens when capacity exceeds the interpolation threshold.

## Model-Wise Double Descent

The **interpolation threshold** is the model capacity at which the model can first achieve zero training error — i.e., perfectly memorize the training set. In the classical regime (below this threshold), the model is underparameterized and cannot fully fit the training data; test error decreases as capacity grows and bias shrinks. At the threshold, the model is forced to produce a unique interpolating solution, and this solution is typically the worst-generalizing one in the interpolating family — it is brittle, sensitive to noise, and achieves peak test error. Beyond the threshold, however, there are many interpolating solutions, and the optimization algorithm (gradient descent) selects among them. Strikingly, the solution selected by gradient descent in overparameterized models tends to have lower test error than the interpolating solution at the threshold.

This produces the characteristic double descent shape: a first U-curve in the underparameterized regime, a peak at the interpolation threshold, and a second descent in the overparameterized regime. Formally, letting $\hat{f}_k$ denote the model of size $k$ (parameterized by, say, width), the model-wise double descent is:

$$
k \mapsto \mathcal{R}_{\text{test}}(\hat{f}_k)
\quad\text{has two descent phases separated by a peak at }\,
k^* = \arg\min_k\, \mathcal{R}_{\text{train}}(\hat{f}_k) = 0.
$$

We can understand why the interpolation threshold is the worst point. Among all interpolating solutions, gradient descent (with small weight initialization and no explicit regularization) converges to the **minimum-norm interpolator**. In the linear case this is provably the pseudoinverse solution:

$$
\hat{\theta} = X^\top (X X^\top)^{-1} y,
$$

where $X \in \mathbb{R}^{N \times d}$ is the design matrix and $y \in \mathbb{R}^N$ are the labels. When $d$ is only slightly larger than $N$ (just past the threshold), the minimum-norm solution still has large norm because the matrix $XX^\top$ is nearly singular — the solution must work hard to interpolate. As $d \to \infty,$ the minimum-norm solution's norm decreases and the solution becomes smoother, explaining the second descent.

**Label noise.** The double descent effect is substantially more pronounced when training labels are noisy. With clean labels, the interpolating solution at the threshold may still generalize reasonably because there is a consistent signal to interpolate. With noisy labels, the model at the threshold must memorize contradictory examples, producing highly non-smooth interpolants that generalize poorly. Nakkiran et al. [@nakkiran2020] use 15% uniform label noise on CIFAR-10 to make the effect clearly visible. We follow this convention.

**Three regimes.** It is useful to distinguish: (1) the **underparameterized** regime where the model cannot fit the training data and test error is determined by the bias-variance tradeoff; (2) the **interpolation threshold** where the model just barely memorizes training data and generalization is worst; and (3) the **overparameterized** regime where there are many interpolating solutions and gradient descent finds the well-generalizing minimum-norm one.

:::{.callout-note}
The double descent curve does not violate the bias-variance tradeoff — it *extends* it. Within the underparameterized regime, the classical U-curve holds exactly. The new insight is that the story does not end at the interpolation threshold: a second, qualitatively different regime begins where overparameterization acts as an implicit regularizer.

:::

**Figure.** We now produce the model-wise double descent curve. The code trains ResNet-18 variants with a width multiplier $k$ sweeping from a small underparameterized network to a large overparameterized one, recording the final train and test error at each width. This figure loads from precomputed results; see [§ Reproducing Nakkiran et al.](#reproducing-nakkiran-et-al.-on-resnet-cifar-10) for the full training code.

In [ ]:
#| label: fig-model-wise-double-descent
#| fig-cap: "Model-wise double descent on CIFAR-10 (15% label noise) with ResNet-18 at varying widths. Train error (blue) falls to zero at the interpolation threshold; test error (orange) peaks there and descends again as the model grows larger. The three regimes are annotated."
#| code-fold: true

# Results from the sweep in §4 (width multiplier k vs. final error).
# Loaded here to keep this section self-contained.
# Replace with results_model_wise computed in §4 if running the full sweep.
try:
    results_model_wise
except NameError:
    # Placeholder: illustrative values matching the qualitative shape
    # from Nakkiran et al. Figure 1. Replace after running §4.
    k_vals    = np.array([0.25, 0.5, 1.0, 2.0, 4.0, 8.0])
    n_params  = np.array([0.28e6, 1.1e6, 4.4e6, 17.6e6, 70.4e6, 281e6])
    train_err = np.array([0.38, 0.18, 0.02, 0.0, 0.0, 0.0])
    test_err  = np.array([0.45, 0.35, 0.55, 0.42, 0.33, 0.27])
    interp_idx = 2   # k=1.0 is near threshold in this illustration
else:
    k_vals    = np.array(results_model_wise["k_vals"])
    n_params  = np.array(results_model_wise["n_params"])
    train_err = 1.0 - np.array(results_model_wise["train_acc"])
    test_err  = 1.0 - np.array(results_model_wise["test_acc"])
    interp_idx = int(results_model_wise["interp_idx"])

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(n_params, train_err * 100, color="C0", linewidth=2, marker="o", ms=5, label="train error")
ax.plot(n_params, test_err  * 100, color="C1", linewidth=2, marker="o", ms=5, label="test error")

# Mark interpolation threshold
ax.axvline(n_params[interp_idx], color="gray", linestyle="dashed", lw=1.2,
           label="interpolation threshold")

# Annotate three regimes
ax.text(n_params[0] * 1.05, 52, "underparameterized", fontsize=8, color="gray", va="top")
ax.text(n_params[interp_idx] * 1.05, 52, "overparameterized", fontsize=8, color="gray", va="top")
ax.annotate("threshold",
            xy=(n_params[interp_idx], test_err[interp_idx] * 100),
            xytext=(n_params[interp_idx] * 2, test_err[interp_idx] * 100 + 4),
            fontsize=8, color="gray",
            arrowprops=dict(arrowstyle="->", color="gray", lw=0.8))

ax.set_xscale("log")
ax.set_xlabel("number of parameters (log scale)")
ax.set_ylabel("error (%)")
ax.set_ylim(0, 60)
ax.legend()
ax.grid(linestyle="dotted", alpha=0.6)
fig.tight_layout()
plt.show();

## Epoch-Wise Double Descent

The double descent phenomenon has a temporal analog that is, if anything, more practically important than the model-wise version. Nakkiran et al. [@nakkiran2020] showed that for a **fixed model** near the interpolation threshold, the test error trace over training epochs can itself exhibit a double descent: test error initially decreases, then *increases* for a prolonged period (as the model approaches the interpolation regime mid-training), then decreases again as training continues further.

To understand this, note that as training progresses, the effective capacity of the model increases in a functional sense — early in training, the model is far from memorizing the training set and behaves like an underparameterized model. Around the epoch where training error first reaches zero, the model enters the interpolation regime with respect to its training trajectory. If the model's architectural capacity is close to the interpolation threshold, this crossing happens at an unfavorable point. The learned representation at that epoch corresponds to the worst-generalizing interpolating solution.

This has a stark practical consequence: for models near the interpolation threshold, **early stopping based on validation loss can be severely harmful**. The standard rule — stop training when validation loss first increases — would terminate training precisely at the worst moment. For such models, training should continue well past the local minimum in validation loss. For models that are comfortably overparameterized or comfortably underparameterized, epoch-wise double descent does not occur and the standard early stopping heuristic is safe.

:::{.callout-caution}
**Early stopping near the interpolation threshold can increase test error.** The conventional wisdom of stopping when validation loss rises is based on the classical regime, where test error is unimodal in training time. For models near the critical capacity, the validation curve has a local minimum followed by a rise and then a further descent. Stopping at the local minimum captures the worst-generalizing model. When in doubt, train longer.

:::

**Figure.** Test error vs. epoch for three ResNet widths: a small model (underparameterized), a critical model (near the interpolation threshold), and a large model (overparameterized). Only the critical model exhibits a visible bump:

In [ ]:
#| label: fig-epoch-wise-double-descent
#| fig-cap: "Epoch-wise double descent. The critical-capacity model (orange) exhibits a pronounced bump in test error around the epoch where training error first reaches zero. The small model (blue) shows a monotone decrease; the large model (green) decreases smoothly throughout. Results from the sweep in §4."
#| code-fold: true

# Loaded from §4 training results. Fallback to illustrative curves.
try:
    results_epoch_wise
except NameError:
    epochs = np.arange(1, 201)
    # Small model: monotone decrease after initial drop, no bump
    small_err    = 0.48 * np.exp(-epochs / 80) + 0.42
    # Critical model: bump around epoch 80-120
    bump         = 0.08 * np.exp(-((epochs - 100) ** 2) / (2 * 20 ** 2))
    critical_err = 0.50 * np.exp(-epochs / 60) + 0.30 + bump
    # Large model: smooth descent
    large_err    = 0.52 * np.exp(-epochs / 50) + 0.25
    small_label    = r"$k = 0.25$ (underparameterized)"
    critical_label = r"$k = 1.0$ (critical)"
    large_label    = r"$k = 4.0$ (overparameterized)"
else:
    epochs         = np.array(results_epoch_wise["epochs"])
    small_err      = 1.0 - np.array(results_epoch_wise["small_test_acc"])
    critical_err   = 1.0 - np.array(results_epoch_wise["critical_test_acc"])
    large_err      = 1.0 - np.array(results_epoch_wise["large_test_acc"])
    small_label    = results_epoch_wise["small_label"]
    critical_label = results_epoch_wise["critical_label"]
    large_label    = results_epoch_wise["large_label"]

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(epochs, small_err    * 100, color="C0", linewidth=2, label=small_label)
ax.plot(epochs, critical_err * 100, color="C1", linewidth=2, label=critical_label)
ax.plot(epochs, large_err    * 100, color="C2", linewidth=2, label=large_label)

# Annotate the bump in the critical model
bump_epoch = epochs[np.argmax(critical_err[50:]) + 50]
bump_val   = critical_err[np.argmax(critical_err[50:]) + 50] * 100
ax.annotate("epoch-wise\ndouble descent",
            xy=(bump_epoch, bump_val),
            xytext=(bump_epoch + 25, bump_val + 3),
            fontsize=8, color="C1",
            arrowprops=dict(arrowstyle="->", color="C1", lw=0.8))

ax.set_xlabel("epoch")
ax.set_ylabel("test error (%)")
ax.legend(fontsize=8)
ax.grid(linestyle="dotted", alpha=0.6)
fig.tight_layout()
plt.show();

## Reproducing Nakkiran et al. on ResNet / CIFAR-10

We now develop the full experimental setup for the width sweep. The key ingredients are: (1) a **width-scalable ResNet-18** parameterized by a multiplier $k$ that scales every stage's channel count; (2) **CIFAR-10 with 15% uniform label noise** applied at dataset construction time (not at the model level); (3) a **fixed 200-epoch training schedule** with cosine annealing so that every model sees the same number of gradient steps; and (4) evaluation on the **clean** test set (noise is training-only).

**Why label noise matters.** Without label noise the interpolation threshold effect is muted because the training labels are consistent — a model at the threshold can find an interpolating solution that generalizes reasonably. With 15% noise, the threshold model must memorize contradictory labels (the same or similar images with different labels), producing a highly irregular function that generalizes poorly. The noise makes the peak in test error much sharper and more visible.

**Width multiplier.** We modify ResNet-18 by replacing the channel counts $\{64, 128, 256, 512\}$ with $\{\lfloor 64k \rfloor, \lfloor 128k \rfloor, \lfloor 256k \rfloor, \lfloor 512k \rfloor\}$ for multiplier $k.$ Small $k$ produces underparameterized models; $k = 1$ is the standard ResNet-18; large $k$ gives overparameterized models.

**Optimizer.** We use SGD with momentum (as in the original paper) rather than AdamW, since SGD's implicit bias toward minimum-norm solutions is part of what drives the second descent. Cosine annealing decays from a peak learning rate of $0.1$ to near zero over 200 epochs.

**Computational note.** Training 6 models for 200 epochs each on CIFAR-10 requires a GPU. The full sweep takes approximately 2–4 hours on a single A100 or RTX 3090. Results are cached in `artifacts/double_descent_results.pkl` so the sweep runs only once; subsequent executions load from the cache.

**Data.** CIFAR-10 with 15% uniform label noise, applied once at construction:

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)
LABEL_NOISE  = 0.15
BATCH_SIZE   = 128


class NoisyCIFAR10(Dataset):
    """CIFAR-10 with a fixed fraction of uniformly corrupted labels."""

    def __init__(self, root, train, transform, noise_rate, seed=0):
        self.dataset = torchvision.datasets.CIFAR10(
            root=root, train=train, download=True, transform=transform
        )
        self.targets = list(self.dataset.targets)         # <1>

        if noise_rate > 0 and train:
            rng = np.random.default_rng(seed)
            n = len(self.targets)
            noisy = rng.choice(n, size=int(noise_rate * n), replace=False)  # <2>
            for i in noisy:
                orig = self.targets[i]
                choices = [c for c in range(10) if c != orig]
                self.targets[i] = int(rng.choice(choices))  # <3>

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        img, _ = self.dataset[idx]                        # <4>
        return img, self.targets[idx]


transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_set = NoisyCIFAR10(
    DATASET_DIR, train=True, transform=transform_train,
    noise_rate=LABEL_NOISE, seed=RANDOM_SEED
)
test_set = torchvision.datasets.CIFAR10(
    root=DATASET_DIR, train=False, download=True, transform=transform_test
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Sanity check: count corrupted labels
orig_targets  = train_set.dataset.targets
noisy_targets = train_set.targets
n_changed = sum(a != b for a, b in zip(orig_targets, noisy_targets))
print(f"Train: {len(train_set)}, Test: {len(test_set)}")
print(f"Labels changed: {n_changed} ({n_changed / len(train_set) * 100:.1f}%)")

1. We copy `dataset.targets` so we can modify labels without corrupting the underlying dataset object.
2. We sample without replacement so the noise rate is exact (not a Bernoulli approximation).
3. Each corrupted label is drawn uniformly at random from the 9 other classes, never the correct class.
4. We discard the label from the underlying dataset and return our (possibly corrupted) label instead.

**Model.** ResNet-18 with a width multiplier $k$ scaling all channel counts:

In [ ]:
class BasicBlock(nn.Module):
    """Residual block with two 3x3 convolutions and a skip connection."""
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(
            out_channels, out_channels, 3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return F.relu(out)


class WideResNet18(nn.Module):
    """ResNet-18 with a width multiplier k for CIFAR-10."""

    def __init__(self, k=1.0, num_classes=10):
        super().__init__()
        c = [max(1, int(64  * k)),
             max(1, int(128 * k)),
             max(1, int(256 * k)),
             max(1, int(512 * k))]               # <1>
        self.in_channels = c[0]

        self.conv1   = nn.Conv2d(3, c[0], 3, stride=1, padding=1, bias=False)  # <2>
        self.bn1     = nn.BatchNorm2d(c[0])
        self.layer1  = self._make_layer(c[0], 2, stride=1)
        self.layer2  = self._make_layer(c[1], 2, stride=2)
        self.layer3  = self._make_layer(c[2], 2, stride=2)
        self.layer4  = self._make_layer(c[3], 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc      = nn.Linear(c[3], num_classes)

    def _make_layer(self, out_channels, num_blocks, stride):
        downsample = None
        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        layers = [BasicBlock(self.in_channels, out_channels, stride, downsample)]
        self.in_channels = out_channels
        for _ in range(1, num_blocks):
            layers.append(BasicBlock(self.in_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))  # (B, c0, 32, 32)
        x = self.layer1(x)                    # (B, c0, 32, 32)
        x = self.layer2(x)                    # (B, c1, 16, 16)
        x = self.layer3(x)                    # (B, c2,  8,  8)
        x = self.layer4(x)                    # (B, c3,  4,  4)
        x = self.avgpool(x)                   # (B, c3,  1,  1)
        x = torch.flatten(x, 1)               # (B, c3)
        return self.fc(x)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


# Preview parameter counts across the width sweep
K_VALS = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0]
for k in K_VALS:
    m = WideResNet18(k=k)
    print(f"k={k:4.2f}  params={count_params(m):>10,}")

1. Each stage's channel count is multiplied by $k$ and clipped to at least 1. The four base channel counts $\{64, 128, 256, 512\}$ are preserved at $k = 1.$
2. CIFAR-10 uses a $3 \times 3$ stem with stride 1 (no initial max-pooling), following the convention from NB12. This preserves the $32 \times 32$ spatial resolution into the first stage.

**Training.** Self-contained training functions using SGD with momentum and cosine annealing:

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total


@torch.inference_mode()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total


def train_model(model, train_loader, test_loader, epochs, device,
                lr=0.1, weight_decay=5e-4, record_every=5, verbose=True):
    """Train model and return a history dict with train/test accuracy traces."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-4)  # <1>

    history = {
        "epoch": [], "train_acc": [], "test_acc": [],
        "train_loss": [], "test_loss": [],
    }

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        scheduler.step()                                  # <2>

        if epoch % record_every == 0 or epoch == 1:
            test_loss, test_acc = evaluate(model, test_loader, criterion, device)
            history["epoch"].append(epoch)
            history["train_acc"].append(train_acc)
            history["test_acc"].append(test_acc)
            history["train_loss"].append(train_loss)
            history["test_loss"].append(test_loss)

            if verbose:
                print(
                    f"[{epoch:>3d}/{epochs}]  "
                    f"train_acc={train_acc:.4f}  test_acc={test_acc:.4f}  "
                    f"lr={scheduler.get_last_lr()[0]:.5f}"
                )

    return history

1. Cosine annealing decays the learning rate from `lr` to `eta_min` over `T_max` epochs, giving each model a consistent schedule regardless of width.
2. The scheduler steps once per epoch (after each full pass over training data), not per batch.

Running the width sweep (GPU strongly recommended; takes ~2–4 hours on a single GPU). Results are cached to `artifacts/double_descent_results.pkl` and loaded on subsequent runs:

In [ ]:
#| output: false

EPOCHS       = 200
LR           = 0.1
WEIGHT_DECAY = 5e-4
RESULTS_PATH = ARTIFACTS_DIR / "double_descent_results.pkl"

if RESULTS_PATH.exists():                                  # <1>
    with open(RESULTS_PATH, "rb") as f:
        sweep_histories = pickle.load(f)
    print(f"Loaded cached results from {RESULTS_PATH}")
else:
    sweep_histories = {}  # k -> history dict

    for k in K_VALS:
        print(f"\n=== k={k} ===")
        torch.manual_seed(RANDOM_SEED)
        model = WideResNet18(k=k)
        history = train_model(
            model, train_loader, test_loader, EPOCHS, DEVICE,
            lr=LR, weight_decay=WEIGHT_DECAY, record_every=5, verbose=True
        )
        sweep_histories[k] = history

    with open(RESULTS_PATH, "wb") as f:                    # <2>
        pickle.dump(sweep_histories, f)
    print(f"\nSweep complete. Results saved to {RESULTS_PATH}")

1. If the pickle file already exists, we load the cached histories rather than re-running the full sweep. This makes re-execution cheap.
2. After completing the sweep, results are persisted so that the figure cells in §2 and §3 can load them directly.

Packaging results for the figure cells in §2 and §3:

In [ ]:
#| output: false

# Model-wise results: final train/test accuracy for each k
final_train_acc = [sweep_histories[k]["train_acc"][-1] for k in K_VALS]
final_test_acc  = [sweep_histories[k]["test_acc"][-1]  for k in K_VALS]
n_params_list   = [count_params(WideResNet18(k=k)) for k in K_VALS]

# Interpolation threshold: first k where train_acc > 0.995 (train_err -> 0)
interp_idx = next(
    (i for i, acc in enumerate(final_train_acc) if acc > 0.995),
    len(K_VALS) - 1
)

results_model_wise = {
    "k_vals":     K_VALS,
    "n_params":   n_params_list,
    "train_acc":  final_train_acc,
    "test_acc":   final_test_acc,
    "interp_idx": interp_idx,
}

# Epoch-wise results: three representative k values
k_small    = K_VALS[0]            # underparameterized
k_critical = K_VALS[interp_idx]   # at interpolation threshold
k_large    = K_VALS[-1]           # overparameterized

results_epoch_wise = {
    "epochs":            sweep_histories[k_small]["epoch"],
    "small_test_acc":    sweep_histories[k_small]["test_acc"],
    "critical_test_acc": sweep_histories[k_critical]["test_acc"],
    "large_test_acc":    sweep_histories[k_large]["test_acc"],
    "small_label":       f"k={k_small} (underparameterized)",
    "critical_label":    f"k={k_critical} (critical)",
    "large_label":       f"k={k_large} (overparameterized)",
}

print(f"Interpolation threshold: k={K_VALS[interp_idx]}, params={n_params_list[interp_idx]:,}")
for k, tr, te in zip(K_VALS, final_train_acc, final_test_acc):
    print(f"  k={k:4.2f}  train_err={1 - tr:.3f}  test_err={1 - te:.3f}")

**Setup.** The phase diagram below (from the original paper) visualizes both model-wise and epoch-wise double descent simultaneously — the $x$-axis is model size and the $y$-axis is number of training epochs. Each cell is colored by test error. The characteristic cross-shaped high-error region (dark) corresponds to the interpolation threshold in both dimensions: high test error whenever the model is near critical width *or* near the critical epoch.

![Nakkiran et al. (2019), Figure 8: phase diagram of test error as a function of model size (x-axis) and training epochs (y-axis). Dark regions = high error; the cross-shaped high-error region corresponds to the interpolation threshold in both dimensions.](img/nakkiran-phase-diagram.png){#fig-nakkiran-phase-diagram width=80%}

*Source: Nakkiran et al., "Deep Double Descent: Where Bigger Models and More Data Hurt", ICLR 2020. Reproduced for educational purposes.*

## Reconciliation with Modern Practice

The double descent phenomenon raises an immediate question: why do overparameterized models generalize at all? If a model has more parameters than data points, it can represent infinitely many functions that interpolate the training data — why does gradient descent find a good one?

**Implicit regularization.** The answer lies in the implicit bias of the optimization algorithm. Gradient descent initialized near zero converges, in the overparameterized regime, to the **minimum-norm interpolator** — the interpolating solution with the smallest parameter norm. For linear models this is provably equal to the pseudoinverse solution:

$$
\hat{\theta} = X^\top (X X^\top)^{-1} y,
$$

where $X \in \mathbb{R}^{N \times d}$ is the design matrix. For $d \gg N,$ this solution tends to be smoother and to generalize better than the minimum-norm solution in the underparameterized regime. The inductive bias of SGD in neural networks plays an analogous role, though the precise characterization is an active research area.

**Weight decay interaction.** Explicit $L_2$ regularization (weight decay) interacts with double descent in a subtle way. Weight decay shifts the interpolation threshold: it makes the effective model capacity smaller, moving the threshold to a larger nominal model size. This means that for a fixed architecture, increasing weight decay can decrease test error at the threshold by pushing the model into the underparameterized regime — or it can improve test error in the overparameterized regime by strengthening the implicit bias toward minimum-norm solutions. The interaction is non-monotone and depends on the noise level.

**Data augmentation.** Augmentation effectively increases the number of training examples, which raises the interpolation threshold. This is one reason aggressive augmentation tends to benefit large models more than small ones: large models are pushed further into the overparameterized regime by the augmented effective dataset size.

**Practical implications.** Several concrete lessons emerge from the double descent picture. For model size, the classical advice to avoid overparameterization is wrong in the modern regime. A larger model is not necessarily worse — it may generalize better than a model near the threshold, provided training is carried out to convergence. The practical strategy is to use a model large enough to be comfortably overparameterized, not to tune capacity to the classical sweet spot.

For early stopping, the epoch-wise double descent shows that stopping at the first minimum of validation loss can be harmful for models near the interpolation threshold. When training a model of unknown position relative to the threshold, it is safer to use a learning rate schedule that reaches near zero (so training terminates naturally) than to rely on early stopping as the termination criterion.

For label noise, the double descent effect is substantially worse with noisy labels, and the peak at the threshold is sharper. Cleaning training data reduces the severity of the interpolation threshold peak and generally improves performance throughout.

**Remark.** Zhang et al. [@zhang2017] showed that deep networks can perfectly memorize randomly labeled training data, achieving zero training error even when labels are pure noise. This demonstrates that the standard model of generalization — which assumed that networks could not simply memorize — was incomplete. Double descent provides the updated picture: memorization does happen, but in the overparameterized regime the minimum-norm interpolator that gradient descent finds tends to be one that also generalizes well on clean labels.

:::{.callout-note}
The double descent phenomenon is not a reason to discard statistical learning theory. The bias-variance tradeoff is correct and tight within the underparameterized regime. Double descent extends the picture: a second descent exists in the overparameterized regime, driven by the implicit minimum-norm bias of gradient descent. Modern deep learning operates almost exclusively in this second regime.

:::

---

■